# MADIVA Workspace: Metadata Exploration Demo
This notebook demonstrates how to authenticate with the Gen3 Data Commons API and extract patient metadata directly into a Pandas environment for analysis.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from gen3.auth import Gen3Auth
from gen3.submission import Gen3Submission
import io

# Authenticate using the injected workspace token
auth = Gen3Auth(refresh_file="/home/jovyan/.gen3/credentials.json")
sub = Gen3Submission(auth)

ModuleNotFoundError: No module named 'matplotlib'

## Extracting and Merging Clinical Nodes
In Gen3, metadata is stored as a graph. We will pull data from two connected nodes: `subject` (core identifiers) and `demographic` (traits), and merge them into a single dataset.

In [ ]:
program = "MADIVA"
project = "Agincourt"

# Export the Subject node
subject_tsv = sub.export_node(program, project, "subject", "tsv")
df_subject = pd.read_csv(io.StringIO(subject_tsv), sep='\t')

# Export the Demographic node
demo_tsv = sub.export_node(program, project, "demographic", "tsv")
df_demo = pd.read_csv(io.StringIO(demo_tsv), sep='\t')

# Merge the datasets on the submitter_id
df_merged = pd.merge(df_subject, df_demo, left_on='submitter_id', right_on='subjects.submitter_id', how='inner')

display(df_merged.head())

## Metadata Summary
Even without raw data files, we can generate summary statistics across the project cohort based purely on the clinical metadata submitted to the graph.

In [ ]:
# Generate a simple visualization using a metadata field (e.g., 'sex' or 'ethnicity')
target_field = 'sex'

if target_field in df_merged.columns:
    df_merged[target_field].value_counts().plot(
        kind='pie', 
        autopct='%1.1f%%', 
        colors=['#4C72B0', '#55A868', '#C44E52']
    )
    plt.title(f"Cohort Distribution by {target_field.capitalize()}")
    plt.ylabel('')
    plt.show()
else:
    print(f"Field '{target_field}' not found. Available columns: {list(df_merged.columns)}")